# LeetCode 3123 — Find Edges in Shortest Paths bằng Python

- Link: https://leetcode.com/problems/find-edges-in-shortest-paths/
- Độ khó: **Hard**
- Chủ đề: **Dijkstra hai lần** (từ `0` và từ `n − 1`); cạnh `(a, b, w)` nằm trên **một** đường ngắn nhất ⇔ `d₀[a] + w + d₁[b] = D` hoặc `d₀[b] + w + d₁[a] = D`; hoặc dựng **DAG đường ngắn nhất** rồi truy ngược từ đích

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def findAnswer(self, n: int, edges: List[List[int]]) -> List[bool]:
```

## 1. Mục tiêu học

1. Dijkstra với heap cho đồ thị trọng số dương `O((n + m) log n)`.
2. Một đỉnh `v` nằm trên đường ngắn nhất `0 → n−1` ⇔ `d₀[v] + d₁[v] = D` (với `D = d₀[n−1]`).
3. Một cạnh `a — b` (vô hướng, trọng số `w`) nằm trên đường ngắn nhất ⇔ đi theo **một trong hai chiều** khớp: `d₀[a] + w + d₁[b] = D` hoặc `d₀[b] + w + d₁[a] = D`.
4. Xử lý đồ thị **không liên thông**: nếu `n − 1` không tới được thì mọi đáp án `False` (tránh `∞ = ∞`).
5. Cách khác: chỉ một lần Dijkstra rồi truy ngược trên DAG "cạnh chặt" (`d₀[a] + w = d₀[b]`) từ `n − 1`.

## 2. Đề bài & ví dụ

Đồ thị vô hướng có trọng số, `n` đỉnh, danh sách cạnh `[a, b, w]`. Xét **mọi** đường đi ngắn nhất từ `0` tới `n − 1`. Trả về mảng `answer`, `answer[i] = True` nếu cạnh `i` thuộc **ít nhất một** đường ngắn nhất. Đồ thị có thể không liên thông.

**Ví dụ 1:** `n = 6`, `edges = [[0,1,4],[0,2,1],[1,3,2],[1,4,3],[1,5,1],[2,3,1],[3,5,3],[4,5,2]]` → `[T,T,T,F,T,T,T,F]`.

**Ví dụ 2:** `n = 4`, `edges = [[2,0,1],[0,1,1],[0,3,4],[3,2,2]]` → `[T,F,F,T]`.

**Ràng buộc:** `2 ≤ n ≤ 5·10⁴`; `1 ≤ m ≤ min(5·10⁴, n(n−1)/2)`; `1 ≤ w ≤ 10⁵`; không có cạnh lặp.

In [1]:
from typing import List
import heapq, itertools, random, time

T, F = True, False
examples = [(6, [[0, 1, 4], [0, 2, 1], [1, 3, 2], [1, 4, 3], [1, 5, 1], [2, 3, 1], [3, 5, 3], [4, 5, 2]], [T, T, T, F, T, T, T, F]),
            (4, [[2, 0, 1], [0, 1, 1], [0, 3, 4], [3, 2, 2]], [T, F, F, T])]
for n, e, exp in examples:
    print('n =', n, '| m =', len(e), '-> kỳ vọng', exp)

n = 6 | m = 8 -> kỳ vọng [True, True, True, False, True, True, True, False]
n = 4 | m = 4 -> kỳ vọng [True, False, False, True]


## 3. Kiến thức nền

### 3.1. Dijkstra

In [2]:
INF = float('inf')
def build(n, edges):
    g = [[] for _ in range(n)]
    for a, b, w in edges:
        g[a].append((b, w)); g[b].append((a, w))
    return g

def dijkstra(g, s):
    d = [INF] * len(g); d[s] = 0; pq = [(0, s)]
    while pq:
        du, u = heapq.heappop(pq)
        if du > d[u]:
            continue
        for v, w in g[u]:
            nd = du + w
            if nd < d[v]:
                d[v] = nd; heapq.heappush(pq, (nd, v))
    return d

g = build(*examples[0][:2])
print('d0 =', dijkstra(g, 0)); print('d1 =', dijkstra(g, 5))

d0 = [0, 4, 1, 2, 7, 5]
d1 = [5, 1, 4, 3, 2, 0]


### 3.2. Điều kiện cạnh trên đường ngắn nhất

Đường `0 ⇝ a → b ⇝ n−1` có độ dài ngắn nhất có thể `d₀[a] + w + d₁[b]` (đoạn đầu và đoạn cuối lấy ngắn nhất). Cạnh nằm trên **một** đường ngắn nhất ⇔ giá trị này bằng `D` (theo chiều `a → b` hoặc `b → a`).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hình dung `d₀` như "độ cao" tính từ nguồn và `d₁` từ đích. Cạnh "chặt" là cạnh mà đi qua nó không lãng phí: tổng khoảng cách còn lại khớp đúng `D`. Ví dụ 1: `D = 5`; cạnh `1 — 4` (`w = 3`): `d₀[1] + 3 + d₁[4] = 4 + 3 + 2 = 9 ≠ 5` và chiều ngược `d₀[4] + 3 + d₁[1] = 7 + 3 + 1 ≠ 5` ⇒ `False`.

## 5. Cách 1 — Brute Force: liệt kê mọi đường đi đơn

In [3]:
def brute(n, edges):
    g = [[] for _ in range(n)]
    for i, (a, b, w) in enumerate(edges):
        g[a].append((b, w, i)); g[b].append((a, w, i))
    best = [INF]; used_sets = []
    def dfs(u, dist, vis, used):
        if dist > best[0]:
            return
        if u == n - 1:
            if dist < best[0]:
                best[0] = dist; used_sets.clear()
            used_sets.append(set(used)); return
        for v, w, i in g[u]:
            if v not in vis:
                vis.add(v); used.append(i)
                dfs(v, dist + w, vis, used)
                used.pop(); vis.remove(v)
    dfs(0, 0, {0}, [])
    on = set().union(*used_sets) if used_sets else set()
    return [i in on for i in range(len(edges))]

for n, e, exp in examples:
    print('brute:', brute(n, e), '| kỳ vọng', exp)

brute: [True, True, True, False, True, True, True, False] | kỳ vọng [True, True, True, False, True, True, True, False]
brute: [True, False, False, True] | kỳ vọng [True, False, False, True]


Đường ngắn nhất với trọng số dương luôn là đường **đơn** ⇒ liệt kê đường đơn là đủ. Mũ theo `n` — chỉ cho đồ thị nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

```
d0 = dijkstra(0); d1 = dijkstra(n-1); D = d0[n-1]
nếu D = ∞: tất cả False
answer[i] = (d0[a] + w + d1[b] == D) or (d0[b] + w + d1[a] == D)
```

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Dijkstra hai lần (lời giải chính)

In [4]:
def two_dijkstra(n, edges):
    g = build(n, edges); d0 = dijkstra(g, 0); d1 = dijkstra(g, n - 1); D = d0[n - 1]
    if D == INF:
        return [False] * len(edges)
    return [d0[a] + w + d1[b] == D or d0[b] + w + d1[a] == D for a, b, w in edges]

for n, e, exp in examples:
    print('2 Dijkstra:', two_dijkstra(n, e), '| kỳ vọng', exp)

2 Dijkstra: [True, True, True, False, True, True, True, False] | kỳ vọng [True, True, True, False, True, True, True, False]
2 Dijkstra: [True, False, False, True] | kỳ vọng [True, False, False, True]


### 7.2. Cách 3 — Một Dijkstra + truy ngược DAG cạnh chặt

Cạnh `a → b` là **chặt** nếu `d₀[a] + w = d₀[b]`. Mọi đường ngắn nhất tới `n − 1` gồm toàn cạnh chặt. Từ `n − 1`, BFS ngược theo cạnh chặt: cạnh `a → b` với `b` đã được đánh dấu "trên đường ngắn nhất" thì cạnh thuộc đáp án và `a` cũng được đánh dấu.

In [5]:
def dag_backtrack(n, edges):
    g = [[] for _ in range(n)]
    for i, (a, b, w) in enumerate(edges):
        g[a].append((b, w, i)); g[b].append((a, w, i))
    d0 = [INF] * n; d0[0] = 0; pq = [(0, 0)]
    while pq:
        du, u = heapq.heappop(pq)
        if du > d0[u]:
            continue
        for v, w, _ in g[u]:
            if du + w < d0[v]:
                d0[v] = du + w; heapq.heappush(pq, (du + w, v))
    ans = [False] * len(edges)
    if d0[n - 1] == INF:
        return ans
    seen = [False] * n; seen[n - 1] = True; stack = [n - 1]
    while stack:
        b = stack.pop()
        for a, w, i in g[b]:
            if d0[a] + w == d0[b]:                    # cạnh chặt a -> b
                ans[i] = True
                if not seen[a]:
                    seen[a] = True; stack.append(a)
    return ans

for n, e, exp in examples:
    print('truy ngược DAG:', dag_backtrack(n, e), '| kỳ vọng', exp)

truy ngược DAG: [True, True, True, False, True, True, True, False] | kỳ vọng [True, True, True, False, True, True, True, False]
truy ngược DAG: [True, False, False, True] | kỳ vọng [True, False, False, True]


### 7.3. Cách 4 — Floyd–Warshall (đồ thị nhỏ)

`dist[u][v]` mọi cặp `O(n³)`; điều kiện cạnh như cách 2 với `d₀ = dist[0]`, `d₁ = dist[n−1]`. Dùng làm oracle thứ hai, độc lập với Dijkstra.

In [6]:
def floyd(n, edges):
    d = [[INF] * n for _ in range(n)]
    for i in range(n):
        d[i][i] = 0
    for a, b, w in edges:
        d[a][b] = min(d[a][b], w); d[b][a] = min(d[b][a], w)
    for k in range(n):
        dk = d[k]
        for i in range(n):
            dik = d[i][k]
            if dik == INF:
                continue
            di = d[i]
            for j in range(n):
                if dik + dk[j] < di[j]:
                    di[j] = dik + dk[j]
    D = d[0][n - 1]
    if D == INF:
        return [False] * len(edges)
    return [d[0][a] + w + d[b][n - 1] == D or d[0][b] + w + d[a][n - 1] == D for a, b, w in edges]

for n, e, exp in examples:
    print('Floyd:', floyd(n, e), '| kỳ vọng', exp)

Floyd: [True, True, True, False, True, True, True, False] | kỳ vọng [True, True, True, False, True, True, True, False]
Floyd: [True, False, False, True] | kỳ vọng [True, False, False, True]


## 8. TRACE từng bước

In `d₀`, `d₁` và hai tổng kiểm tra cho từng cạnh của ví dụ 1.

In [7]:
n, e, _ = examples[0]
g = build(n, e); d0 = dijkstra(g, 0); d1 = dijkstra(g, n - 1); D = d0[n - 1]
print('D =', D)
for i, (a, b, w) in enumerate(e):
    s1 = d0[a] + w + d1[b]; s2 = d0[b] + w + d1[a]
    print(f'cạnh {i}: {a} —{w}— {b}: d0[a]+w+d1[b] = {s1}, d0[b]+w+d1[a] = {s2} -> {s1 == D or s2 == D}')

D = 5
cạnh 0: 0 —4— 1: d0[a]+w+d1[b] = 5, d0[b]+w+d1[a] = 13 -> True
cạnh 1: 0 —1— 2: d0[a]+w+d1[b] = 5, d0[b]+w+d1[a] = 7 -> True
cạnh 2: 1 —2— 3: d0[a]+w+d1[b] = 9, d0[b]+w+d1[a] = 5 -> True
cạnh 3: 1 —3— 4: d0[a]+w+d1[b] = 9, d0[b]+w+d1[a] = 11 -> False
cạnh 4: 1 —1— 5: d0[a]+w+d1[b] = 5, d0[b]+w+d1[a] = 7 -> True
cạnh 5: 2 —1— 3: d0[a]+w+d1[b] = 5, d0[b]+w+d1[a] = 7 -> True
cạnh 6: 3 —3— 5: d0[a]+w+d1[b] = 5, d0[b]+w+d1[a] = 11 -> True
cạnh 7: 4 —2— 5: d0[a]+w+d1[b] = 9, d0[b]+w+d1[a] = 9 -> False


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def findAnswer(self, n: int, edges: List[List[int]]) -> List[bool]:
        g = [[] for _ in range(n)]
        for a, b, w in edges:
            g[a].append((b, w))
            g[b].append((a, w))
        INF = float('inf')

        def dijkstra(s):
            d = [INF] * n
            d[s] = 0
            pq = [(0, s)]
            while pq:
                du, u = heapq.heappop(pq)
                if du > d[u]:
                    continue
                for v, w in g[u]:
                    nd = du + w
                    if nd < d[v]:
                        d[v] = nd
                        heapq.heappush(pq, (nd, v))
            return d

        d0 = dijkstra(0)
        D = d0[n - 1]
        if D == INF:
            return [False] * len(edges)
        d1 = dijkstra(n - 1)
        return [d0[a] + w + d1[b] == D or d0[b] + w + d1[a] == D for a, b, w in edges]

for n, e, exp in examples:
    got = Solution().findAnswer(n, e)
    print(got, 'OK' if got == exp else 'SAI')

[True, True, True, False, True, True, True, False] OK
[True, False, False, True] OK


## 10. Vì sao thuật toán đúng

1. (⇐) Nếu `d₀[a] + w + d₁[b] = D`: nối đường ngắn nhất `0 ⇝ a`, cạnh `a → b`, đường ngắn nhất `b ⇝ n−1` được một đường độ dài `D` — ngắn nhất (có thể không đơn? Với trọng số dương, đường độ dài `D` là ngắn nhất nên không thể lặp đỉnh — lặp sẽ tạo chu trình độ dài dương có thể bỏ, cho đường `< D`, mâu thuẫn).
2. (⇒) Nếu cạnh nằm trên một đường ngắn nhất theo chiều `a → b`: đoạn trước `≥ d₀[a]`, đoạn sau `≥ d₁[b]`, tổng `= D ≤ d₀[a] + w + d₁[b]` ⇒ mọi bất đẳng thức là đẳng thức.
3. **Không liên thông:** `D = ∞` ⇒ không có đường nào ⇒ mọi cạnh `False` (phải xử lý riêng vì `∞ + w + x == ∞` đúng trong Python).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê đường đơn | mũ | `O(n + m)` |
| 2. Hai Dijkstra | `O((n + m) log n)` | `O(n + m)` |
| 3. Một Dijkstra + DAG | `O((n + m) log n)` | `O(n + m)` |
| 4. Floyd–Warshall | `O(n³)` | `O(n²)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'two_dijkstra': two_dijkstra, 'dag': dag_backtrack, 'floyd': floyd,
         'submit': lambda n, e: Solution().findAnswer(n, e)}
edge = [(2, [[0, 1, 5]], [T]), (3, [[0, 1, 2]], [F]), (3, [[0, 1, 1], [1, 2, 1], [0, 2, 2]], [T, T, T]),
        (4, [[0, 1, 1], [2, 3, 1]], [F, F]), (4, [[0, 1, 1], [1, 3, 1], [0, 2, 1], [2, 3, 2]], [T, T, F, F])]
for n, e, exp in examples + edge:
    assert brute(n, e) == exp, (n, e, brute(n, e))
    for name, f in impls.items():
        assert f(n, e) == exp, (name, n, e)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `(3, [[0, 1, 2]])` — đỉnh 2 cô lập ⇒ không có đường ⇒ `False`. Tam giác `0–1–2` với `0–2` nặng 2 ⇒ hai đường ngắn nhất cùng độ dài 2 dùng cả ba cạnh.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [10]:
rng = random.Random(3123)
def rand_graph(n, m, W):
    pairs = list(itertools.combinations(range(n), 2)); rng.shuffle(pairs)
    return [[a, b, rng.randint(1, W)] if rng.random() < 0.5 else [b, a, rng.randint(1, W)] for a, b in pairs[:m]]

cnt = 0
for _ in range(300):
    n = rng.randint(2, 7); m = rng.randint(1, n * (n - 1) // 2)
    e = rand_graph(n, m, rng.choice([1, 2, 5]))
    exp = brute(n, e)
    for name, f in impls.items():
        assert f(n, e) == exp, (name, n, e)
    cnt += 1
print('Khớp brute trên', cnt, 'đồ thị ngẫu nhiên (trọng số nhỏ ⇒ nhiều đường ngắn nhất bằng nhau).')

Khớp brute trên 300 đồ thị ngẫu nhiên (trọng số nhỏ ⇒ nhiều đường ngắn nhất bằng nhau).


### 12.3. Ca lớn `n = m = 5·10⁴`

In [11]:
N = 5 * 10 ** 4
def big_graph(n, m):
    es = set(); out = []
    for v in range(1, n):                                   # cây ngẫu nhiên đảm bảo liên thông
        u = rng.randint(max(0, v - 50), v - 1); es.add((u, v)); out.append([u, v, rng.randint(1, 10 ** 5)])
    while len(out) < m:
        a, b = rng.randrange(n), rng.randrange(n)
        if a != b and (min(a, b), max(a, b)) not in es:
            es.add((min(a, b), max(a, b))); out.append([a, b, rng.randint(1, 10 ** 5)])
    return out

cases = {'liên thông, trọng số lớn': big_graph(N, N),
         'trọng số 1 (nhiều đường bằng nhau)': [[a, b, 1] for a, b, _ in big_graph(N, N)],
         'không liên thông': [[i, i + 1, 1] for i in range(N - 2)]}
for name, e in cases.items():
    t = time.time(); x = Solution().findAnswer(N, e); t1 = time.time() - t
    y = dag_backtrack(N, e)
    assert x == y
    print(f'{name:36s}: {sum(x):6d} cạnh True / {len(e)} | nộp {t1:.2f}s')

liên thông, trọng số lớn            :   1365 cạnh True / 50000 | nộp 0.13s
trọng số 1 (nhiều đường bằng nhau)  :   1407 cạnh True / 50000 | nộp 0.13s


không liên thông                    :      0 cạnh True / 49998 | nộp 0.08s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm một chiều `d₀[a] + w + d₁[b]`

In [12]:
def wrong_one_direction(n, edges):
    g = build(n, edges); d0 = dijkstra(g, 0); d1 = dijkstra(g, n - 1); D = d0[n - 1]
    if D == INF:
        return [False] * len(edges)
    return [d0[a] + w + d1[b] == D for a, b, w in edges]               # SAI: cạnh vô hướng

print('sai :', wrong_one_direction(*examples[1][:2]), '| đúng:', Solution().findAnswer(*examples[1][:2]))

sai : [False, False, False, False] | đúng: [True, False, False, True]


**Vì sao sai:** cạnh được cho dạng `[2, 0, 1]` nhưng đường ngắn nhất đi `0 → 2`. Đồ thị vô hướng ⇒ kiểm cả hai chiều.

### Lỗi 2 — Không xử lý trường hợp không tới được `n − 1`

In [13]:
def wrong_unreachable(n, edges):
    g = build(n, edges); d0 = dijkstra(g, 0); d1 = dijkstra(g, n - 1); D = d0[n - 1]
    return [d0[a] + w + d1[b] == D or d0[b] + w + d1[a] == D for a, b, w in edges]   # SAI khi D = inf

args = (4, [[0, 1, 1], [2, 3, 1]])
print('sai :', wrong_unreachable(*args), '| đúng:', Solution().findAnswer(*args))

sai : [True, True] | đúng: [False, False]


**Vì sao sai:** `inf + 1 + 0 == inf` là `True` trong Python ⇒ mọi cạnh có một đầu không tới được sẽ bị đánh dấu nhầm.

### Lỗi 3 — Chỉ khôi phục **một** đường ngắn nhất (mảng cha)

In [14]:
def wrong_single_path(n, edges):
    g = [[] for _ in range(n)]
    for i, (a, b, w) in enumerate(edges):
        g[a].append((b, w, i)); g[b].append((a, w, i))
    d = [INF] * n; d[0] = 0; par = [None] * n; pq = [(0, 0)]
    while pq:
        du, u = heapq.heappop(pq)
        if du > d[u]: continue
        for v, w, i in g[u]:
            if du + w < d[v]:
                d[v] = du + w; par[v] = (u, i); heapq.heappush(pq, (du + w, v))
    ans = [False] * len(edges)
    if d[n - 1] == INF: return ans
    v = n - 1
    while par[v] is not None:
        u, i = par[v]; ans[i] = True; v = u                                     # SAI: chỉ một đường
    return ans

print('sai :', wrong_single_path(*examples[0][:2]), '| đúng:', Solution().findAnswer(*examples[0][:2]))

sai : [False, True, False, False, False, True, True, False] | đúng: [True, True, True, False, True, True, True, False]


**Vì sao sai:** ví dụ 1 có **ba** đường ngắn nhất; mảng cha chỉ giữ một.

### Lỗi 4 — Dùng BFS (bỏ qua trọng số)

In [15]:
from collections import deque
def wrong_bfs(n, edges):
    g = [[] for _ in range(n)]
    for a, b, w in edges:
        g[a].append(b); g[b].append(a)
    def bfs(s):
        d = [INF] * n; d[s] = 0; q = deque([s])
        while q:
            u = q.popleft()
            for v in g[u]:
                if d[v] == INF: d[v] = d[u] + 1; q.append(v)
        return d
    d0, d1 = bfs(0), bfs(n - 1); D = d0[n - 1]
    if D == INF: return [False] * len(edges)
    return [d0[a] + 1 + d1[b] == D or d0[b] + 1 + d1[a] == D for a, b, w in edges]   # SAI

print('sai :', wrong_bfs(*examples[1][:2]), '| đúng:', Solution().findAnswer(*examples[1][:2]))

sai : [False, False, True, False] | đúng: [True, False, False, True]


**Vì sao sai:** BFS tìm đường ít **cạnh** nhất (`0 → 3` trực tiếp), không phải nhẹ nhất (`0 → 2 → 3`, tổng 3 < 4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cạnh/đỉnh nằm trên **một** đường ngắn nhất" ⇒ Dijkstra từ hai đầu + điều kiện tổng bằng `D`.
- "Cạnh nằm trên **mọi** đường ngắn nhất" ⇒ thêm điều kiện cầu trong DAG đường ngắn nhất (đếm số đường).
- Luôn kiểm trường hợp không liên thông khi so sánh với `∞`.

## 15. Biến thể của bài

1. Cạnh nằm trên **mọi** đường ngắn nhất ⇔ `cnt₀[a] · cnt₁[b] = cnt₀[n−1]` (đếm số đường ngắn nhất, nên làm modulo vài số nguyên tố).
2. Đồ thị **có hướng** ⇒ chỉ một chiều, `d₁` tính trên đồ thị đảo.
3. Trọng số `0` cho phép ⇒ vẫn đúng với Dijkstra nhưng "đường ngắn nhất" có thể không đơn.
4. Xoá một cạnh, hỏi đường ngắn nhất mới có tăng không ⇒ cạnh tăng ⇔ nằm trên mọi đường (biến thể 1).

In [16]:
# Biến thể 1: cạnh nằm trên MỌI đường ngắn nhất (đếm số đường, số nguyên Python chính xác)
def on_all_paths(n, edges):
    g = build(n, edges)
    def dij_count(s):
        d = [INF] * n; c = [0] * n; d[s] = 0; c[s] = 1; pq = [(0, s)]; done = [False] * n
        while pq:
            du, u = heapq.heappop(pq)
            if done[u]: continue
            done[u] = True
            for v, w in g[u]:
                if du + w < d[v]:
                    d[v] = du + w; c[v] = c[u]; heapq.heappush(pq, (d[v], v))
                elif du + w == d[v]:
                    c[v] += c[u]
        return d, c
    (d0, c0), (d1, c1) = dij_count(0), dij_count(n - 1); D = d0[n - 1]; total = c0[n - 1]
    if D == INF: return [False] * len(edges)
    res = []
    for a, b, w in edges:
        k = 0
        if d0[a] + w + d1[b] == D: k += c0[a] * c1[b]
        if d0[b] + w + d1[a] == D: k += c0[b] * c1[a]
        res.append(k == total)
    return res

def on_all_brute(n, edges):
    g = [[] for _ in range(n)]
    for i, (a, b, w) in enumerate(edges):
        g[a].append((b, w, i)); g[b].append((a, w, i))
    paths = []; best = [INF]
    def dfs(u, dist, vis, used):
        if u == n - 1:
            if dist < best[0]: best[0] = dist; paths.clear()
            if dist == best[0]: paths.append(set(used))
            return
        for v, w, i in g[u]:
            if v not in vis:
                vis.add(v); used.append(i); dfs(v, dist + w, vis, used); used.pop(); vis.remove(v)
    dfs(0, 0, {0}, [])
    if not paths: return [False] * len(edges)
    common = set.intersection(*paths)
    return [i in common for i in range(len(edges))]

for _ in range(200):
    n = rng.randint(2, 6); e = rand_graph(n, rng.randint(1, n * (n - 1) // 2), rng.choice([1, 2, 3]))
    assert on_all_paths(n, e) == on_all_brute(n, e), (n, e)
print('Biến thể "trên mọi đường ngắn nhất": khớp brute.')

Biến thể "trên mọi đường ngắn nhất": khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Liệt kê đường đơn | mũ | Oracle |
| 2 | Hai Dijkstra | `O((n+m) log n)` | **Nộp bài** |
| 3 | Một Dijkstra + DAG ngược | `O((n+m) log n)` | Ít phép Dijkstra hơn |
| 4 | Floyd–Warshall | `O(n³)` | Oracle độc lập |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 3`, `edges = [[0,1,1],[1,2,1],[0,2,3]]`.

**Bài 2 (tính tay):** `n = 4`, `edges = [[0,1,2],[0,2,2],[1,3,2],[2,3,2],[1,2,1]]`.

**Bài 3 (điền TODO):** trả về danh sách **đỉnh** nằm trên ít nhất một đường ngắn nhất.

**Bài 4 (tự viết lại):** phiên bản đồ thị **có hướng**.

**Bài 5 (mini challenge):** đếm số đường ngắn nhất `0 → n−1` mod `10^9 + 7` (LC 1976).

In [17]:
def vertices_on_paths_todo(n, edges):
    # TODO
    raise NotImplementedError

def directed_version_todo(n, edges):
    # TODO
    raise NotImplementedError

def count_shortest_paths_todo(n, edges):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `D = 2` qua đỉnh 1. (2) Cạnh `0–2` nặng 3 > 2. (3) `[True, True, False]`.

**Bài 2.** (1) `D = 4`: hai đường `0-1-3`, `0-2-3`. (2) Cạnh `1–2` (`w = 1`): `d₀[1] + 1 + d₁[2] = 2 + 1 + 2 = 5 > 4`. (3) `[True, True, True, True, False]`.

**Bài 3.** (1) `d₀[v] + d₁[v] = D`. (2) Bỏ qua khi `D = ∞`. (3) Kiểm bằng hợp các đỉnh của đường ngắn nhất (brute).

**Bài 4.** (1) `d₁` tính trên đồ thị **đảo chiều** từ `n − 1`. (2) Chỉ kiểm `d₀[a] + w + d₁[b]`. (3) Brute trên đồ thị có hướng.

**Bài 5.** (1) Dijkstra kèm đếm: khi `nd < d[v]` gán `cnt[v] = cnt[u]`, khi bằng thì cộng. (2) Xử lý mỗi đỉnh một lần (`done`). (3) Mod.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().findAnswer(3, [[0, 1, 1], [1, 2, 1], [0, 2, 3]]) == [True, True, False]
assert Solution().findAnswer(4, [[0, 1, 2], [0, 2, 2], [1, 3, 2], [2, 3, 2], [1, 2, 1]]) == [True, True, True, True, False]

def vertices_on_paths(n, edges):
    g = build(n, edges); d0 = dijkstra(g, 0); d1 = dijkstra(g, n - 1); D = d0[n - 1]
    if D == INF: return []
    return [v for v in range(n) if d0[v] + d1[v] == D]

for _ in range(200):
    n = rng.randint(2, 7); e = rand_graph(n, rng.randint(1, n * (n - 1) // 2), rng.choice([1, 2, 5]))
    on = brute(n, e)
    verts = sorted({x for i, (a, b, w) in enumerate(e) if on[i] for x in (a, b)})
    if not verts and n >= 1 and dijkstra(build(n, e), 0)[n - 1] == 0:
        verts = [0]
    assert vertices_on_paths(n, e) == verts
print('Bài 1, 2 đúng | Bài 3: đỉnh trên đường ngắn nhất khớp brute.')

Bài 1, 2 đúng | Bài 3: đỉnh trên đường ngắn nhất khớp brute.


In [19]:
# Bài 4, 5
def directed_version(n, edges):
    g = [[] for _ in range(n)]; rg = [[] for _ in range(n)]
    for a, b, w in edges:
        g[a].append((b, w)); rg[b].append((a, w))
    d0 = dijkstra(g, 0); d1 = dijkstra(rg, n - 1); D = d0[n - 1]
    if D == INF: return [False] * len(edges)
    return [d0[a] + w + d1[b] == D for a, b, w in edges]

def directed_brute(n, edges):
    g = [[] for _ in range(n)]
    for i, (a, b, w) in enumerate(edges): g[a].append((b, w, i))
    best = [INF]; sets = []
    def dfs(u, dist, vis, used):
        if u == n - 1:
            if dist < best[0]: best[0] = dist; sets.clear()
            if dist == best[0]: sets.append(set(used))
            return
        for v, w, i in g[u]:
            if v not in vis:
                vis.add(v); used.append(i); dfs(v, dist + w, vis, used); used.pop(); vis.remove(v)
    dfs(0, 0, {0}, [])
    on = set().union(*sets) if sets else set()
    return [i in on for i in range(len(edges))]

def count_shortest_paths(n, edges, MOD=10 ** 9 + 7):
    g = build(n, edges); d = [INF] * n; c = [0] * n; d[0] = 0; c[0] = 1; pq = [(0, 0)]; done = [False] * n
    while pq:
        du, u = heapq.heappop(pq)
        if done[u]: continue
        done[u] = True
        for v, w in g[u]:
            if du + w < d[v]: d[v] = du + w; c[v] = c[u]; heapq.heappush(pq, (d[v], v))
            elif du + w == d[v]: c[v] = (c[v] + c[u]) % MOD
    return c[n - 1]

for _ in range(200):
    n = rng.randint(2, 6); e = rand_graph(n, rng.randint(1, n * (n - 1) // 2), rng.choice([1, 2, 3]))
    assert directed_version(n, e) == directed_brute(n, e)
    gg = [[] for _ in range(n)]
    for a, b, w in e: gg[a].append((b, w)); gg[b].append((a, w))
    best = [INF]; cnt = [0]
    def dfs(u, dist, vis):
        if u == n - 1:
            if dist < best[0]: best[0] = dist; cnt[0] = 0
            if dist == best[0]: cnt[0] += 1
            return
        for v, w in gg[u]:
            if v not in vis: vis.add(v); dfs(v, dist + w, vis); vis.remove(v)
    dfs(0, 0, {0})
    assert count_shortest_paths(n, e) == cnt[0]
assert count_shortest_paths(7, [[0,6,7],[0,1,2],[1,2,3],[1,3,3],[6,3,3],[3,5,1],[6,5,1],[2,5,1],[0,4,5],[4,6,2]]) == 4   # LC 1976
print('Bài 4 (có hướng) và Bài 5 (đếm đường ngắn nhất, LC 1976) khớp brute.')

Bài 4 (có hướng) và Bài 5 (đếm đường ngắn nhất, LC 1976) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
d0 = dijkstra(0); D = d0[n-1]
if D == inf: return [False] * m
d1 = dijkstra(n-1)
return [d0[a] + w + d1[b] == D or d0[b] + w + d1[a] == D for a, b, w in edges]
```

**Checklist:**

- [ ] Dijkstra (không BFS) vì có trọng số?
- [ ] Kiểm **cả hai chiều** của cạnh vô hướng?
- [ ] `D = ∞` ⇒ trả toàn `False`?
- [ ] Đánh dấu **mọi** đường ngắn nhất, không chỉ một?